In [1]:
# Import library
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity

print("Memulai Fase 3 (Versi Optimal): Perakitan dan Pengujian Sistem Hybrid")

# --- Langkah 3.1 (Revisi): Muat Komponen yang Benar ---
try:
    # Data olahan
    user_profile_df = pd.read_csv('../dataset/processed_user_profiles.csv')
    user_skill_matrix = pd.read_csv('../dataset/processed_user_skills.csv')

    # Otak dari mesin A (Content-Based) - SEKARANG KITA LOAD PROFILNYA
    encoded_profiles = joblib.load('../models/content_based_encoded_profiles.joblib')

    # Otak dari mesin B (Collaborative)
    skill_similarity_df = joblib.load('../models/collaborative_skill_similarity.joblib')
    print("Semua data dan model berhasil dimuat.")
except FileNotFoundError:
    print("Error: Pastikan model dari Fase 2 (yang sudah direvisi) sudah disimpan.")
    exit()

# --- Langkah 3.2 (Revisi): Perbaiki Fungsi untuk Mesin A ---

# Fungsi untuk Mesin A (Content-Based) - VERSI BARU ON-THE-FLY
def recommend_by_profile(user_id, all_encoded_profiles, top_n_users=50, top_n_skills=10):
    """
    Menghitung kemiripan on-the-fly untuk satu user, lalu merekomendasikan skill.
    Ini jauh lebih hemat memori.
    """
    # Ambil vektor profil dari user target
    target_user_vector = all_encoded_profiles[user_id]
    
    # HITUNG KEMIRIPAN ON-THE-FLY: antara 1 user dengan semua user lain
    similarity_scores = cosine_similarity(target_user_vector, all_encoded_profiles)[0]
    
    # Lanjutan logikanya sama seperti sebelumnya
    similar_user_indices = np.argsort(similarity_scores)[-top_n_users-1:-1][::-1]
    
    similar_users_skills = user_skill_matrix.iloc[similar_user_indices]
    skill_popularity = similar_users_skills.drop(columns=['user_id']).sum().sort_values(ascending=False)
    
    user_known_skills_series = user_skill_matrix[user_skill_matrix['user_id'] == user_id]
    if not user_known_skills_series.empty:
        user_known_skills = user_known_skills_series.drop(columns=['user_id'])
        skills_to_remove = user_known_skills.columns[user_known_skills.iloc[0] == 1]
        final_recommendations = skill_popularity.drop(skills_to_remove, errors='ignore')
    else:
        final_recommendations = skill_popularity
        
    return final_recommendations.head(top_n_skills)

# Fungsi untuk Mesin B (Collaborative) - TIDAK ADA PERUBAHAN
def recommend_by_skills(known_skills, top_n_skills=10):
    all_recommendations = pd.Series(dtype=float)
    for skill in known_skills:
        if skill in skill_similarity_df.columns:
            recommendations = skill_similarity_df[skill].sort_values(ascending=False)
            all_recommendations = pd.concat([all_recommendations, recommendations])
    final_recommendations = all_recommendations.groupby(all_recommendations.index).sum()
    final_recommendations = final_recommendations.drop(known_skills, errors='ignore')
    return final_recommendations.sort_values(ascending=False).head(top_n_skills)


# --- Langkah 3.3: Buat Fungsi Hybrid (Dengan Perbaikan Kecil) ---
# Tambahkan pengecekan agar tidak error saat normalisasi jika hanya ada 1 rekomendasi

def hybrid_recommender(user_id, weight_content=0.7, weight_collab=0.3, top_n=10):
    # Dapatkan rekomendasi dari kedua mesin
    content_recs = recommend_by_profile(user_id, encoded_profiles) # Kirim encoded_profiles
    
    user_known_skills_series = user_skill_matrix[user_skill_matrix['user_id'] == user_id]
    known_skills_list = []
    if not user_known_skills_series.empty:
        series = user_known_skills_series.drop(columns=['user_id']).iloc[0]
        known_skills_list = series[series == 1].index.tolist()
        
    collab_recs = recommend_by_skills(known_skills_list)
    
    # Normalisasi skor (dengan pengecekan)
    if not content_recs.empty and (content_recs.max() - content_recs.min()) > 0:
        content_recs_normalized = (content_recs - content_recs.min()) / (content_recs.max() - content_recs.min())
    else:
        content_recs_normalized = content_recs

    if not collab_recs.empty and (collab_recs.max() - collab_recs.min()) > 0:
        collab_recs_normalized = (collab_recs - collab_recs.min()) / (collab_recs.max() - collab_recs.min())
    else:
        collab_recs_normalized = collab_recs
        
    # Gabungkan skor dengan bobot
    final_scores = pd.Series(dtype=float)
    for skill, score in content_recs_normalized.items():
        final_scores[skill] = score * weight_content
    for skill, score in collab_recs_normalized.items():
        if skill in final_scores:
            final_scores[skill] += score * weight_collab
        else:
            final_scores[skill] = score * weight_collab
            
    return final_scores.sort_values(ascending=False).head(top_n)


# --- Langkah 3.4: UJI COBA! ---
test_user_id = 100

print(f"\n--- Melakukan Uji Coba untuk User ID: {test_user_id} ---")
print("\nProfil User:")
print(user_profile_df[user_profile_df['user_id'] == test_user_id])

user_known_skills_series = user_skill_matrix[user_skill_matrix['user_id'] == test_user_id]
known_skills_list = []
if not user_known_skills_series.empty:
    series = user_known_skills_series.drop(columns=['user_id']).iloc[0]
    known_skills_list = series[series == 1].index.tolist()
print(f"\nSkill yang Sudah Dikuasai ({len(known_skills_list)}):")
print(known_skills_list)

print("\n--- REKOMENDASI HYBRID (WEIGHTED) ---")
final_recommendations = hybrid_recommender(user_id=test_user_id)
print(final_recommendations)

print("\n--- FASE 3 SELESAI (VERSI OPTIMAL) ---")

Memulai Fase 3 (Versi Optimal): Perakitan dan Pengujian Sistem Hybrid
Semua data dan model berhasil dimuat.

--- Melakukan Uji Coba untuk User ID: 100 ---

Profil User:
     user_id    age gender    country  \
100      100  18-21  Woman  Singapore   

                                             education job_title  \
100  Some college/university study without earning ...   Student   

    years_experience  
100        < 1 years  

Skill yang Sudah Dikuasai (3):
['R', ' Google Cloud TPUs ', ' Ggplot / ggplot2 ']

--- REKOMENDASI HYBRID (WEIGHTED) ---
Python                              1.000000
 Matplotlib                         0.401004
Linear or Logistic Regression       0.386842
Decision Trees or Random Forests    0.227185
  Scikit-learn                      0.209432
 Seaborn                            0.186417
Coursera                            0.168966
Kaggle Learn Courses                0.168966
DataCamp                            0.096552
Colab Notebooks                     0.